In [1]:
import pandas as pd
import os

# Select certain pow files

In [2]:
POW_FOLDER = "../data/pow_bus1to8"
SELECTED_FILES = [
    "uov_pow_1.csv",
    "uov_pow_1_rev.csv",
    # "uov_pow_2.csv",
    # "uov_pow_2_rev.csv",
    "uov_pow_3.csv",
    "uov_pow_3_rev.csv",
    # "uov_pow_5.csv",
    # "uov_pow_5_rev.csv",
    # "uov_pow_6.csv",
    # "uov_pow_6_rev.csv",
    # "uov_pow_7.csv",
    # "uov_pow_7_rev.csv",
    "uov_pow_28.csv", # core route
    "uov_pow_28_rev.csv",
    "uov_pow_73.csv", # core route
    "uov_pow_73_rev.csv",
    # "uov_pow_77.csv", # core route
    # "uov_pow_77_rev.csv"
]
pow_files = [os.path.join(POW_FOLDER, f) for f in SELECTED_FILES]

# Import pow

In [3]:
import re

pow_df_list = []

for file_path in pow_files:
    df = pd.read_csv(file_path)

    filename = os.path.basename(file_path)
    match = re.search(r'uov_pow_(\d+)(\_rev)?', filename)
    route_num = int(match.group(1))
    is_rev = match.group(2) is not None
    route_id = -route_num if is_rev else route_num

    df['route_id'] = route_id
    pow_df_list.append(df)
 
pow_df_raw = pd.concat(pow_df_list, ignore_index=True)

print("raw shape:", pow_df_raw.shape)
pow_df_raw.head()

raw shape: (1013, 9)


,trip_index,start_stop,end_stop,start_hour,start_min,end_hour,end_min,duration_min,route_id
0,1,IJsselsteinlaan,Vechtsebanen,6,3,6,44,41,1
1,2,IJsselsteinlaan,Vechtsebanen,6,18,6,59,41,1
2,3,IJsselsteinlaan,Vechtsebanen,6,32,7,14,42,1
3,4,IJsselsteinlaan,Vechtsebanen,6,47,7,29,42,1
4,5,IJsselsteinlaan,Vechtsebanen,6,55,7,39,44,1


In [4]:
pow_df_raw = pow_df_raw.copy()
pow_df_raw["pow_id"] = range(1, len(pow_df_raw) + 1)
stations = pd.concat([pow_df_raw["start_stop"],pow_df_raw['end_stop']]).unique()
station_to_id = {station: i for i, station in enumerate(stations)}
pow_df_raw["start_stop_id"] = pow_df_raw["start_stop"].map(station_to_id)
pow_df_raw["end_stop_id"] = pow_df_raw["end_stop"].map(station_to_id)
pow_df_raw.head()

,trip_index,start_stop,end_stop,start_hour,start_min,end_hour,end_min,duration_min,route_id,pow_id,start_stop_id,end_stop_id
0,1,IJsselsteinlaan,Vechtsebanen,6,3,6,44,41,1,1,0,1
1,2,IJsselsteinlaan,Vechtsebanen,6,18,6,59,41,1,2,0,1
2,3,IJsselsteinlaan,Vechtsebanen,6,32,7,14,42,1,3,0,1
3,4,IJsselsteinlaan,Vechtsebanen,6,47,7,29,42,1,4,0,1
4,5,IJsselsteinlaan,Vechtsebanen,6,55,7,39,44,1,5,0,1


# Transform time format into minute

In [5]:
def prepare_pow_input(pow_df_raw):
    df = pow_df_raw.copy()

    df["start_time"] = df["start_hour"] * 60 + df["start_min"]
    df["end_time"] = df["end_hour"] * 60 + df["end_min"]

    pow_df_temp = df[[
        "pow_id",
        "trip_index",
        "start_time",
        "end_time",
        "duration_min",
        "start_stop_id",
        "end_stop_id",
        "route_id"
    ]].copy()

    pow_df = pow_df_temp.sort_values(["pow_id"]).reset_index(drop=True)
    return pow_df

In [6]:
pow_df = prepare_pow_input(pow_df_raw)
print("shape:", pow_df.shape)
pow_df.head()

shape: (1013, 8)


,pow_id,trip_index,start_time,end_time,duration_min,start_stop_id,end_stop_id,route_id
0,1,1,363,404,41,0,1,1
1,2,2,378,419,41,0,1,1
2,3,3,392,434,42,0,1,1
3,4,4,407,449,42,0,1,1
4,5,5,415,459,44,0,1,1


# Parameters

In [7]:
MAX_GAP = 30 # The max gap between two pow
MAX_SPAN = 8 * 60 # The longest time of a duty
MAX_POWS_PER_DUTY = 8 # The max amount of pow in a duty
REQUIRE_SAME_POSITION = True # If next pow should start at the same stop as the end stop of last pow
MAX_WORKING_TIME = 3 * 60
BREAK_TIME = 30
MIN_SPAN = 5 * 60

# Restrictions in generating duties
1. The start time of the next pow should be later than the end time of the last pow
2. The gap between two pows should be between two restrictions
3. The next pow should start at the same stop as the end stop of last pow

In [8]:
def can_follow(current_row, next_row, max_gap=MAX_GAP, require_same_position=REQUIRE_SAME_POSITION):
    # No overlapping
    if next_row["start_time"] < current_row["end_time"]:
        return False
    # No too large or too small gap
    gap = next_row["start_time"] - current_row["end_time"]
    if gap > max_gap:
        return False
    # start next pow from the end stop of last pow
    if require_same_position and current_row["end_stop_id"] != next_row["start_stop_id"]:
        return False
    return True

In [9]:
def should_take_break(current_duty_indices,records,max_work_time=MAX_WORKING_TIME,break_time=BREAK_TIME):
    work_times = [records[i]["duration_min"] for i in current_duty_indices]
    total_driving_time = sum(work_times)
    total_work_time = records[current_duty_indices[-1]]['end_time'] - records[current_duty_indices[0]]['start_time']
    total_break_time = total_work_time - total_driving_time
    if total_driving_time > max_work_time and total_break_time < break_time:
        return True
    return False

# Check if a duty reaches the max span

In [10]:
def exceed_max_span(records, current_duty_indices, next_idx, max_span=MAX_SPAN):
    first_idx = current_duty_indices[0]
    first_start = records[first_idx]["start_time"]
    new_end = records[next_idx]["end_time"]

    return (new_end - first_start) > max_span

In [11]:
def normalize(values, higher_is_better=True):
    min_v, max_v = min(values), max(values)
    if max_v == min_v:
        return [0.5] * len(values)
    if higher_is_better:
        return [(v - min_v) / (max_v - min_v) for v in values]
    else:
        return [(max_v - v) / (max_v - min_v) for v in values]

# Generate the list of the next pow of a pow

In [12]:
def generate_next_indices(current_duty_indices,
                          records,
                          last_idx,
                          max_gap=MAX_GAP,
                          top_k=2,
                          max_work_time = MAX_WORKING_TIME,
                          break_time=BREAK_TIME):
    feasible_next = []
    last_row = records[last_idx]
    for next_idx in range(1, len(records)):
        next_row = records[next_idx]
        if should_take_break(current_duty_indices,records):
            if next_row["start_time"] - last_row["end_time"] < break_time:
                continue
            else:
                if not can_follow(last_row, next_row, max_gap):
                    continue
                if exceed_max_span(records, current_duty_indices, next_idx):
                    continue
        if not can_follow(last_row, next_row, max_gap):
            continue
        if exceed_max_span(records, current_duty_indices, next_idx):
            continue

        feasible_next.append(next_idx)

        durations = [records[i]["duration_min"] for i in feasible_next]
        gaps = [records[i]["start_time"] - last_row["end_time"] for i in feasible_next]

        duration_scores = normalize(durations)
        gap_scores = normalize(gaps,higher_is_better=False)

        scores = [0.5 * d + 0.5 *g for d, g in zip(duration_scores, gap_scores)]

        feasible_next = sorted(feasible_next, key=lambda idx: scores[feasible_next.index(idx)], reverse=True)[:top_k]

    return feasible_next

# From a pow, create all feasible duties

In [13]:
def generate_alternatives(pow_df, root_idx, max_pows_per_duty=MAX_POWS_PER_DUTY):
    records = pow_df.to_dict("records")
    results = []
    stack = [[root_idx]]

    while stack:
        current_path = stack.pop()
        last_idx = current_path[-1]

        feasible_next = generate_next_indices(current_path,records,last_idx)
        can_extend = (len(current_path) < max_pows_per_duty and len(feasible_next) > 0)

        if not can_extend:
            results.append([records[i] for i in current_path])

        else:
            for next_idx in feasible_next:
                new_path = current_path + [next_idx]
                stack.append(new_path)
    return results

# Transform the duty set to  a dataframe

In [14]:
def duties_to_dataframe(duties):
    rows = []
    for duty_id, duty in enumerate(duties,start=1):
        rows.append({
            "duty_id": duty_id,
            "pow_ids": [row["pow_id"] for row in duty],
            "trip_indices": [row["trip_index"] for row in duty],
            "start_time": duty[0]["start_time"],
            "end_time": duty[-1]["end_time"],
            "span": duty[-1]["end_time"] - duty[0]["start_time"],
        })
    return pd.DataFrame(rows)

In [15]:
all_duties = []
for root_idx in range(len(pow_df)):
    duties_from_root = generate_alternatives(pow_df, root_idx)
    for duty in duties_from_root:
        # Skip if a duty reaches the min span
        if duty[-1]["end_time"]-duty[0]["start_time"] > MIN_SPAN:
            all_duties.append(duty)
duties_df = duties_to_dataframe(all_duties)
print(f"number of duties: {len(all_duties)}")
duties_df.head()
duties_df.to_csv("duties_output.csv", index=False)

number of duties: 20276


In [16]:
all_pow_ids = sorted(pow_df["pow_id"].to_list())
# Transform dataframe into matrix
coverage_matrix = pd.DataFrame(
    {f"Duty_{row['duty_id']}": [int(pid in set(row["pow_ids"])) for pid in all_pow_ids] for _,row in duties_df.iterrows()},
    index=[f"PoW_{pid}" for pid in all_pow_ids]
)
print("shape of matrix: ",coverage_matrix.shape)
coverage_matrix.to_csv("coverage_matrix_1_3_28_73.csv")

shape of matrix:  (1013, 20276)
